In [1]:
!pip install pandas

In [3]:
!pip3 install -U openai

  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached jiter-0.9.0-cp313-cp313-win_amd64.whl.metadata (5.3 kB)
   ---------------------------------------- 0.0/647.0 kB ? eta -:--:--
   --------------------------------------- 647.0/647.0 kB 14.4 MB/s eta 0:00:00
Using cached distro-1.9.0-py3-none-any.whl (20 kB)
Using cached jiter-0.9.0-cp313-cp313-win_amd64.whl (204 kB)


In [10]:
import pandas as pd
import re
from typing import List, Dict, Any
import os
from openai import OpenAI

class DrugInteractionRetriever:
    def __init__(self, data_dir: str, api_key: str, model: str = "deepseek-chat"):
        """
        Initialize a drug interaction retriever with DeepSeek API integration.
        
        Args:
            data_dir: Directory containing CSV files with drug interaction data
            api_key: DeepSeek API key
            model: DeepSeek model to use (default is "deepseek-chat")
        """
        self.data_dir = data_dir
        self.drug_data = self._load_data()
        self.drug_list = self._create_drug_list()
        
        # Initialize DeepSeek client
        self.model = model
        self.client = OpenAI(api_key=api_key, base_url="https://api.deepseek.com")
        print(f"DeepSeek client initialized with model: {model}")
        
    def _load_data(self) -> pd.DataFrame:
        """Load and combine all CSV files in the data directory."""
        all_dfs = []
        
        for file in os.listdir(self.data_dir):
            if file.endswith('.csv'):
                file_path = os.path.join(self.data_dir, file)
                try:
                    df = pd.read_csv(file_path)
                    all_dfs.append(df)
                    print(f"Loaded {file_path}")
                except Exception as e:
                    print(f"Error loading {file_path}: {e}")
        
        if not all_dfs:
            raise ValueError("No valid CSV files found in the specified directory")
            
        # Combine all dataframes
        combined_df = pd.concat(all_dfs, ignore_index=True)
        
        # Standardize column names (assuming the data has columns for Drug A, Drug B, and interaction level)
        # Adjust these based on your actual column names
        standard_columns = {
            'drug_a': ['drug_a', 'drug1', 'medication_a', 'med_a', 'drug_name_1'],
            'drug_b': ['drug_b', 'drug2', 'medication_b', 'med_b', 'drug_name_2'],
            'interaction_level': ['interaction_level', 'severity', 'interaction_severity', 'level', 'risk_level']
        }
        
        # Standardize column names
        for std_col, possible_names in standard_columns.items():
            for col in combined_df.columns:
                if col.lower() in possible_names:
                    combined_df.rename(columns={col: std_col}, inplace=True)
                    break
                    
        return combined_df
        
    def _create_drug_list(self) -> List[str]:
        """Create a comprehensive list of drugs from the dataset."""
        drugs_a = set(self.drug_data['drug_a'].str.lower().dropna().tolist())
        drugs_b = set(self.drug_data['drug_b'].str.lower().dropna().tolist())
        return list(drugs_a.union(drugs_b))
    
    def extract_drugs_from_query(self, query: str) -> List[str]:
        """Extract drug names from query using simple pattern matching."""
        query_lower = query.lower()
        found_drugs = []
        
        # Use pattern matching with the drug list
        for drug in self.drug_list:
            pattern = r'\b' + re.escape(drug) + r'\b'
            if re.search(pattern, query_lower):
                found_drugs.append(drug)
                
        return found_drugs
    
    def search_interactions(self, drug_list: List[str]) -> pd.DataFrame:
        """
        Search for interactions between the extracted drugs.
        
        Args:
            drug_list: List of extracted drug names
            
        Returns:
            DataFrame containing relevant interactions
        """
        if len(drug_list) < 2:
            # Not enough drugs to check for interactions
            return pd.DataFrame()
            
        # Create all possible drug pairs for checking
        drug_pairs = []
        for i in range(len(drug_list)):
            for j in range(i+1, len(drug_list)):
                drug_pairs.append((drug_list[i], drug_list[j]))
                
        # Search for interactions
        results = []
        for drug_a, drug_b in drug_pairs:
            # Search in both directions (A-B and B-A)
            condition_1 = (
                (self.drug_data['drug_a'].str.lower() == drug_a) & 
                (self.drug_data['drug_b'].str.lower() == drug_b)
            )
            condition_2 = (
                (self.drug_data['drug_a'].str.lower() == drug_b) & 
                (self.drug_data['drug_b'].str.lower() == drug_a)
            )
            
            matches = self.drug_data[condition_1 | condition_2]
            if not matches.empty:
                results.append(matches)
                
        # Combine all results
        if results:
            return pd.concat(results, ignore_index=True)
        else:
            return pd.DataFrame()
    
    def process_query(self, query: str) -> Dict[str, Any]:
        """
        Process a user query and retrieve relevant drug interaction information.
        
        Args:
            query: User's query text
            
        Returns:
            Dictionary containing the original query, extracted drugs, and interaction data
        """
        # Extract drugs from the query
        extracted_drugs = self.extract_drugs_from_query(query)
        
        # If no drugs found, return empty result
        if not extracted_drugs:
            return {
                "original_query": query,
                "extracted_drugs": [],
                "interactions_found": False,
                "interaction_data": []
            }
            
        # Search for interactions
        interaction_df = self.search_interactions(extracted_drugs)
        
        # Format the results
        if interaction_df.empty:
            return {
                "original_query": query,
                "extracted_drugs": extracted_drugs,
                "interactions_found": False,
                "interaction_data": []
            }
            
        # Convert dataframe to list of dictionaries
        interaction_data = interaction_df.to_dict('records')
        
        return {
            "original_query": query,
            "extracted_drugs": extracted_drugs,
            "interactions_found": True,
            "interaction_data": interaction_data
        }
    
    def format_for_llm(self, retriever_output: Dict[str, Any]) -> str:
        """
        Format the retriever output to be sent to DeepSeek.
        
        Args:
            retriever_output: Output from the process_query method
            
        Returns:
            Formatted text to send to the LLM
        """
        query = retriever_output["original_query"]
        drugs = retriever_output["extracted_drugs"]
        interactions_found = retriever_output["interactions_found"]
        interaction_data = retriever_output["interaction_data"]
        
        formatted_text = f"""
User Query: {query}

Extracted Drugs: {', '.join(drugs)}

Drug Interaction Information:
"""
        
        if not interactions_found or not interaction_data:
            formatted_text += "No known interactions found between the extracted drugs."
        else:
            formatted_text += f"Found {len(interaction_data)} potential interaction(s):\n\n"
            
            for i, interaction in enumerate(interaction_data, 1):
                drug_a = interaction.get('drug_a', 'Unknown')
                drug_b = interaction.get('drug_b', 'Unknown')
                level = interaction.get('interaction_level', 'Unknown')
                
                formatted_text += f"Interaction {i}:\n"
                formatted_text += f"- Drug A: {drug_a}\n"
                formatted_text += f"- Drug B: {drug_b}\n"
                formatted_text += f"- Severity: {level}\n"
                
                # Include other details if available
                for key, value in interaction.items():
                    if key not in ['drug_a', 'drug_b', 'interaction_level']:
                        formatted_text += f"- {key}: {value}\n"
                
                formatted_text += "\n"
                
        return formatted_text
    
    def generate_llm_response(self, query: str) -> str:
        """
        Process the user query and generate a response using DeepSeek API.
        
        Args:
            query: User's query about drug interactions
            
        Returns:
            DeepSeek-generated response about potential drug interactions
        """
        # Process the query to retrieve drug interaction data
        retriever_output = self.process_query(query)
        
        # Format the data for LLM input
        formatted_content = self.format_for_llm(retriever_output)
        
        # Create system message with instructions
        system_content = """
You are a helpful assistant providing information about drug interactions. You are not a doctor, and users should always consult with healthcare professionals before making any medication decisions.

Based on the drug interaction data provided, give a clear, informative response about the potential interactions between these drugs. Include severity levels and any important warnings, and remind the user to consult with a healthcare professional before making any medication decisions.
"""
        
        # Send to DeepSeek API
        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": system_content},
                    {"role": "user", "content": formatted_content}
                ],
                stream=False
            )
            
            # Extract the generated text from the response
            return response.choices[0].message.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"


# Example usage
if __name__ == "__main__":
    # Example usage
    data_directory = "./"
    api_key = "API_KEY"  # Replace with your actual API key
    
    # Initialize retriever with DeepSeek API
    retriever = DrugInteractionRetriever(
        data_dir=data_directory,
        api_key=api_key,
        model="deepseek-reasoner"  # Use "deepseek-reasoning" for enhanced reasoning capabilities
    )
    
    # Process a sample query
    sample_query = "Can I take potassium chloride and acrivastine together? I'm also on lisinopril."
    
    # Generate response
    response = retriever.generate_llm_response(sample_query)
    print("\nDeepSeek Response:")
    print(response)
    
    # For debugging or development purposes
    retriever_output = retriever.process_query(sample_query)
    formatted_prompt = retriever.format_for_llm(retriever_output)
    print("\nFormatted input sent to DeepSeek:")
    print(formatted_prompt)

Loaded ./ddinter_downloads_code_A.csv
Loaded ./ddinter_downloads_code_B.csv
Loaded ./ddinter_downloads_code_D.csv
Loaded ./ddinter_downloads_code_H.csv
Loaded ./ddinter_downloads_code_L.csv
Loaded ./ddinter_downloads_code_P.csv
Loaded ./ddinter_downloads_code_R.csv
Loaded ./ddinter_downloads_code_V.csv
DeepSeek client initialized with model: deepseek-reasoner

DeepSeek Response:
**Potential Drug Interactions Between Potassium Chloride, Acrivastine, and Lisinopril**

1. **Lisinopril + Potassium Chloride (Severity: Major)**  
   - **Interaction:** ACE inhibitors like lisinopril can reduce potassium excretion, increasing the risk of **hyperkalemia** (dangerously high potassium levels) when combined with potassium supplements like potassium chloride.  
   - **Warnings:**  
     - Hyperkalemia can cause muscle weakness, irregular heart rhythms, or cardiac arrest.  
     - Regular blood tests to monitor potassium levels are critical.  
     - Avoid potassium-rich diets or salt substitutes un

In [ ]:
sk-e59ac8a298c7470e849a27ddc12cc972